I am coding my way through deep learning history using just numpy to improve my understanding of AI. This is a completed implementation of a basic Transformer language model. I will use it as a base off of which to continue learning by building other deep learning innovations (e.g., improvements to the Transformer like RoPE, Adam, better memory management; RL post-training).

I am using this doc to detail each step of the forward and backward pass to make sure I am truly understanding what I am building: https://docs.google.com/spreadsheets/d/1bDB1wPxcVrq55hjSDb3glXPYFl0f3-6sNImfGcF9huw/edit?usp=sharing

No AI-generated code was used in building the Transformer. The only AI-generated code in this repo is in the quick_testing notebook. I did use AI as a teacher - to validate my understanding or answer questions that arose while reading through papers and articles.

The final version here uses CuPy, so it cannot be run on a CPU. I ran code by connecting to a Colab GPU runtime from my IDE.

## Proof of Concept

### Single set-up only run once per session

In [2]:
!git clone -b main https://github.com/josephwargo/TransformerLanguageModel.git
import sys
sys.path.append('/content/TransformerLanguageModel')

fatal: destination path 'TransformerLanguageModel' already exists and is not an empty directory.


In [ ]:
# stuff I didn't write
import importlib
import numpy as np
import cupy as cp
from datasets import load_dataset
import re
import json
import gc

# stuff I wrote
import Embeddings.positional_embedding as pe
import Layer_Blocks.feed_forward as ff
import Attention.attention_block as ab
import Attention.attention_head as ah
import Layer_Blocks.transformer_block as tb
import Layer_Blocks.layer_norm as ln
import base_transformer as bt
from Helper_Functions import import_word_embeddings as iwe
from Helper_Functions import import_corpus as ic

In [4]:
# loading embeddings from huggingface into cache and then returning where it is located
# NOTE: need to alter this if pulling from local & if we want to use numpy instead of cupy
embeddings_filepath = iwe.download_word_embeddings(repo_id="stanfordnlp/glove", filename_zipped="glove.6B.zip", filename_unzipped='glove.6B.300d.txt')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


glove.6B.zip: reconstructing file:   0%|          |  0.00B /  862MB            

glove.6B.zip: downloading bytes:           |  0.00B            

Extracting text file...


In [5]:
# loading corpus dataset
corpus_dataset = load_dataset("stanfordnlp/imdb")

README.md:   0%|          | 0.00/7.81k [00:00<?, ?B/s]

plain_text/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 21.0MB            

plain_text/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 20.5MB            

plain_text/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

plain_text/unsupervised-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 42.0MB            

plain_text/unsupervised-00000-of-00001.p(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/25000 [00:00<?, ? examples/s]

Generating unsupervised split:   0%|          | 0/50000 [00:00<?, ? examples/s]

In [6]:
# parsing to prep for batching
# constants
start_token = '<START>'
end_token = '<END>'
num_samples = 10000


# loading corpus from huggingface and then parsing
imdb_corpus = ic.parse_corpus(corpus_dataset, start_token, end_token, num_samples)

# getting dictionary of {word: index} for each word in corpus
word2ind = ic.word_to_ind(imdb_corpus, '<PAD>')

# getting list of all words
ind2word = ic.ind_to_word(imdb_corpus, '<PAD>')


# returns a cupy array of the embeddings for our corpus
embeddings = iwe.get_embeddings_for_corpus(embeddings_filepath, word2ind, 300)

In [7]:
# batching, mapping
batch_size = 4
x_batches, Y_batches = ic.batch_corpus(imdb_corpus, word2ind, batch_size=batch_size)


### Proof of Concept

Training the Transformer to predict the next digit given an input of digits - to prove training works (note: the below test code is mostly AI generated)

In [10]:
# import numpy as np
import cupy as np

# Assuming your transformer class is imported as bt
import base_transformer as bt 

def generate_element_wise_data(num_batches, batch_size, seq_len, vocab_size):
    x_batches = []
    Y_batches = []
    
    for _ in range(num_batches):
        # 1. Generate random token indices
        X_indices = np.random.randint(0, vocab_size, size=(batch_size, seq_len))
        
        # 2. Target is simply each token incremented by 1 (wrapped by vocab_size)
        # Shape: (batch_size, seq_len) - This perfectly satisfies your training block
        Y_batch = (X_indices + 1) % vocab_size
        
        x_batches.append(X_indices)
        Y_batches.append(Y_batch.astype(np.int32))
        
    return x_batches, Y_batches

def run_demonstration():
    # Hyperparameters tuned for fast convergence (Proof of Life)
    vocab_size = 20      
    embedding_dim = 64
    seq_len = 4          
    batch_size = 16
    num_batches = 1000   
    
    print("=" * 50)
    print("RUNNING ZERO-MODIFICATION PROOF OF CONCEPT")
    print("=" * 50)
    
    # Mocking the pre-trained embeddings matrix
    embeddings = np.random.normal(0, 0.02, size=(vocab_size, embedding_dim)).astype(np.float32)
    
    # Generate data matching your exact expected matrix shapes
    x_batches, Y_batches = generate_element_wise_data(
        num_batches=num_batches, 
        batch_size=batch_size, 
        seq_len=seq_len, 
        vocab_size=vocab_size
    )
    
    # Initialize your model exactly as it is written
    test_transformer = bt.transformer(
        embeddings=embeddings,
        input_layer_shape=embedding_dim, 
        input_layer_activation='relu',
        d_model=64,                                  
        hidden_layer_activations=['relu', 'relu'],   
        hidden_layer_num_heads=4,
        output_shape=vocab_size,                     
        learning_rate=0.001,                         
        adam=False                                    
    )
    
    # Train the Model
    print("Starting Training Loop...")
    # Your code will handle the flattening internally without crashing
    test_transformer.train(x_batches, Y_batches, num_batches=num_batches)
    
    # Evaluate Verification
    print("\n" + "=" * 50)
    print("VERIFYING INFERENCE (SLICED) RUN")
    print("=" * 50)
    
    test_X, test_Y = generate_element_wise_data(1, 1, seq_len, vocab_size)
    
    # Run a forward pass (triggers your 'else' block, which slices x[:, -1, :])
    logits, _ = test_transformer.forward_pass(test_X[0], train=False)
    
    # Logits shape from your inference block is (1, vocab_size)
    prediction = np.argmax(logits, axis=-1)[0]
    
    # For inference, we only look at the final token because of your slice
    last_input_token = test_X[0][0][-1]
    last_target_token = test_Y[0][0][-1]
    
    print(f"Full Input Sequence: {test_X[0][0]}")
    print(f"Final Input Token:   {last_input_token}")
    print(f"Expected Next Token: {last_target_token}")
    print(f"Model Prediction:    {prediction}")
    
    if prediction == last_target_token:
        print("\nSUCCESS: The model trained and inferred perfectly!")
    else:
        print("\nKeep training: The model has not fully converged yet.")

if __name__ == "__main__":
    run_demonstration()

RUNNING ZERO-MODIFICATION PROOF OF CONCEPT
Starting Training Loop...
Batch: 0
Loss: 3.934141834517358

Batch: 1
Loss: 3.369646186201866

Batch: 2
Loss: 3.755923388881413

Batch: 3
Loss: 3.5120872725794623

Batch: 4
Loss: 3.504002128654351

Batch: 5
Loss: 3.2996753908790426

Batch: 6
Loss: 3.2315640405108548

Batch: 7
Loss: 2.9842773029208

Batch: 8
Loss: 3.1792191588889693

Batch: 9
Loss: 3.0063221924538963

Batch: 10
Loss: 3.1278507140319727

Batch: 11
Loss: 3.1591242388093432

Batch: 12
Loss: 3.040559248788239

Batch: 13
Loss: 2.9545059376990865

Batch: 14
Loss: 3.264692704297406

Batch: 15
Loss: 3.0252148033836055

Batch: 16
Loss: 3.2191153254496694

Batch: 17
Loss: 3.1494254270841124

Batch: 18
Loss: 3.055907148613577

Batch: 19
Loss: 3.090598186087066

Batch: 20
Loss: 3.0150963377854634

Batch: 21
Loss: 2.963056045048962

Batch: 22
Loss: 2.951070735603851

Batch: 23
Loss: 2.9636950233205197

Batch: 24
Loss: 3.0202661044059598

Batch: 25
Loss: 2.9716292091323044

Batch: 26
Loss: 2.

### Transformer Model

Training a small model for 100 batches of real text to show that loss continuously goes down

In [18]:
# transformer model
vocab_size = len(word2ind)
test_transformer = bt.transformer(
      embeddings=embeddings
    , input_layer_shape=300, input_layer_activation='relu'
    , d_model=512, hidden_layer_activations=['relu', 'relu', 'relu']
    , hidden_layer_num_heads=8
    , output_shape=vocab_size
    , clip_val=1
    , learning_rate=.01
    , adam=False
)

In [19]:
test_transformer.train(x_batches=x_batches, Y_batches=Y_batches, num_batches=100)

Batch: 0
Loss: 11.178927544644514

Batch: 1
Loss: 11.155296437030936

Batch: 2
Loss: 11.150755634443831

Batch: 3
Loss: 11.124269076762246

Batch: 4
Loss: 11.128813460011857

Batch: 5
Loss: 11.067843162765657

Batch: 6
Loss: 11.048786698027305

Batch: 7
Loss: 11.026328129129901

Batch: 8
Loss: 11.039671762665527

Batch: 9
Loss: 10.950740166864717

Batch: 10
Loss: 11.056692156793337

Batch: 11
Loss: 11.049766466125831

Batch: 12
Loss: 10.791491159733438

Batch: 13
Loss: 10.812019734836475

Batch: 14
Loss: 10.917600588762017

Batch: 15
Loss: 10.818623510206113

Batch: 16
Loss: 10.815607848630647

Batch: 17
Loss: 10.981821813423728

Batch: 18
Loss: 10.847943209749447

Batch: 19
Loss: 10.7973860008083

Batch: 20
Loss: 10.811434205596155

Batch: 21
Loss: 10.693074155060438

Batch: 22
Loss: 10.566015118669648

Batch: 23
Loss: 10.851695446731158

Batch: 24
Loss: 10.740361848948128

Batch: 25
Loss: 10.512613434607557

Batch: 26
Loss: 10.490214036212686

Batch: 27
Loss: 10.7224116963065

Batch:

## Useful code that is not part of the PoC - saving for later

### Memory management

In [21]:
!nvidia-smi

Sat Jul 18 20:39:02 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   76C    P0             34W /   70W |   14287MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [25]:
del x_batches, Y_batches, embeddings, test_transformer#, padded_corpus

gc.collect()
mempool = cp.get_default_memory_pool()
pinned_mempool = cp.get_default_pinned_memory_pool()
mempool.free_all_blocks()
pinned_mempool.free_all_blocks()


In [26]:
import torch
torch.cuda.empty_cache()

In [28]:
import sys
import gc

def clear_memory_traps():
    # 1. Clear Jupyter/IPython Output History
    try:
        ipy = get_ipython()
        if ipy:
            # Clear the massive hidden output dictionaries
            for key in ['_ih', '_oh', '_dh']:
                if key in ipy.user_ns:
                    ipy.user_ns[key].clear()
            
            # Clear the recent output shortcut variables
            for key in ['_', '__', '___']:
                ipy.user_ns.pop(key, None)
    except NameError:
        pass # Not running in an IPython environment

    # 2. Clear Exception Tracebacks
    # This releases all local variables held at the moment of a crash
    sys.last_type = None
    sys.last_value = None
    sys.last_traceback = None
    
    # If IPython specifically cached a traceback, clear it
    if 'ipy' in locals() and ipy and hasattr(ipy, 'traceback'):
        ipy.traceback = None

    # 3. Clear Metric Logging Lists
    # Replace 'loss_history' with whatever list variable you used to track metrics
    if 'loss_history' in globals():
        globals()['loss_history'].clear()

    # Force garbage collection to execute immediately
    gc.collect()

clear_memory_traps()

### For editing code while running notebook on a GPU via Colab - run every time underlying repo is updated

In [8]:
# Pull the latest changes for your specific branch
%cd /content/TransformerLanguageModel
!git pull origin main
%cd /content

/content/TransformerLanguageModel
From https://github.com/josephwargo/TransformerLanguageModel
 * branch            main       -> FETCH_HEAD
Already up to date.
/content


In [9]:
# # # reloading packages that had any changes
importlib.reload(pe)
importlib.reload(ff)
importlib.reload(ab)
importlib.reload(ah)
importlib.reload(tb)
importlib.reload(ln)
importlib.reload(bt)
importlib.reload(iwe)
importlib.reload(ic)

<module 'Helper_Functions.import_corpus' from '/content/TransformerLanguageModel/Helper_Functions/import_corpus.py'>

### Graveyard

In [ ]:
test_output_list = [int(x) for x in test_output[0]]

# len()
for index in test_output_list:
    print(corpus_words[index])

distanced


In [ ]:
# ONLY USE TO SAVE/LOAD MODELS
# save_file_path = 'C:/Users/josep/Desktop/Self/Learning/NLP/Transformer/Saved_Models/Test_Model_1'
# test_transformer.save_model(save_file_path)
# test_transformer.load_model(save_file_path)

In [ ]:
# test_input = cp.array(
# [
#     [1]
#     , [2]
#     , [3]
#     , [4]
# ]
# )

test_input = cp.array([[3, 4]])

test_output = test_transformer.next_token_vocab_index(test_input)
# word2ind["hello"]